# Chapter 7 — Data Cleaning: The RECAP Framework

Companion notebook for *Data Science for Business and Economics: A Python Approach* ---Jorge Zazueta Gutiérrez.

This notebook mirrors every code example in the chapter and runs as a single sequential pipeline: the cleaned `invoices` table built in **Repair** carries through **Examine**, **Consolidate**, **Align**, and **Prove**. Run the cells in order.

Protagonist: **Daniel Okafor**, spend analyst at **Meridian Components**. Task: turn three incompatible invoice exports into a single, defensible quarterly spend figure by vendor.

Cells are tagged with a `listing` number (matching the chapter's Code Example labels) and a `status`: `frozen` for deterministic cells, `live` for cells that call the network.

In [16]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.width', 100)
pd.set_option('display.max_columns', 20)

# Bound any network call (the one live FX cell) without cluttering it.
import socket
socket.setdefaulttimeout(15)

## Repair: Types and Formats

In [17]:
invoices = pd.read_csv("meridian_invoices.csv")

print(invoices.shape)
print(invoices.dtypes)

(2520, 8)
invoice_id       object
vendor_name      object
currency         object
channel          object
invoice_date     object
amount           object
tax_amount      float64
cost_center      object
dtype: object


In [18]:
print(invoices["amount"].head(6).tolist())

['$5795.79', 'EUR 64,465.43', '$858.44', '$72728.13', 'EUR 34,794.58', '  $28835.30 ']


In [19]:
cleaned = (
    invoices["amount"]
    .str.replace("EUR", "", regex=False)   # symbol, emailed channel
    .str.replace("$", "", regex=False)     # symbol, ERP channels
    .str.replace(",", "", regex=False)     # thousands separator
    .str.strip()                            # stray whitespace
)

invoices["amount"] = pd.to_numeric(cleaned)

print(invoices["amount"].dtype)
print(invoices["amount"].head(3).tolist())

float64
[5795.79, 64465.43, 858.44]


In [20]:
formats = {
    "erp_north":   "%Y-%m-%d",   # 2026-09-09
    "erp_south":   "%m/%d/%Y",   # 08/27/2026
    "email_sheet": "%d-%b-%y",   # 13-Aug-26
}

parsed = pd.Series(pd.NaT, index=invoices.index, dtype="datetime64[ns]")
for channel, fmt in formats.items():
    rows = invoices["channel"] == channel
    parsed[rows] = pd.to_datetime(invoices.loc[rows, "invoice_date"],
                                   format=fmt)

invoices["invoice_date"] = parsed
print(invoices["invoice_date"].isnull().sum())
print(invoices["invoice_date"].min(), "to", invoices["invoice_date"].max())

0
2026-07-01 00:00:00 to 2026-09-30 00:00:00


In [21]:
parsed_mixed = pd.to_datetime(invoices["invoice_date"], format="mixed")
# On Meridian's data this matches the explicit parse exactly.
print((parsed_mixed == invoices["invoice_date"]).all())

True


## Examine: Missing Values and Their Mechanisms

In [22]:
print(invoices[["tax_amount", "cost_center"]].isnull().sum())

tax_amount     126
cost_center    439
dtype: int64


In [23]:
by_channel = (
    invoices.assign(missing=invoices["cost_center"].isnull())
    .groupby("channel")["missing"].mean()
)
print(by_channel.round(3))

channel
email_sheet    0.850
erp_north      0.023
erp_south      0.018
Name: missing, dtype: float64


In [24]:
invoices["cost_center"] = invoices["cost_center"].fillna("Unknown")
print(invoices["cost_center"].value_counts())

cost_center
CC-100     537
CC-200     536
CC-300     530
CC-400     478
Unknown    439
Name: count, dtype: int64


In [25]:
by_channel = (
    invoices.assign(missing=invoices["tax_amount"].isnull())
    .groupby("channel")["missing"].mean()
)
print(by_channel.round(3))

channel
email_sheet    0.062
erp_north      0.050
erp_south      0.045
Name: missing, dtype: float64


In [26]:
reconstructed = (invoices["amount"] * 0.08).round(2)
invoices["tax_amount"] = invoices["tax_amount"].fillna(reconstructed)

print(invoices["tax_amount"].isnull().sum())

0


## Consolidate: Duplicates and Joins

In [27]:
print(invoices.duplicated().sum())
print(invoices["invoice_id"].duplicated().sum())

120
120


In [28]:
repeated_id = invoices.loc[
    invoices["invoice_id"].duplicated(), "invoice_id"
].iloc[0]
print(invoices[invoices["invoice_id"] == repeated_id]
      [["invoice_id", "vendor_name", "channel", "amount"]])

    invoice_id     vendor_name    channel    amount
273  MC-101414  KWAN PRECISION  erp_south  18010.48
405  MC-101414  KWAN PRECISION  erp_south  18010.48


In [29]:
print(f"{invoices['amount'].sum():,.2f}")

invoices = invoices.drop_duplicates()
print(f"{invoices['amount'].sum():,.2f}")

75,646,347.57
72,448,297.35


In [30]:
line_items = pd.read_csv("meridian_line_items.csv")
print(line_items.head(6))
print(len(line_items), line_items["invoice_id"].nunique())

  invoice_id  line_no    sku  quantity  unit_price  line_total
0  MC-100000        1  ST-01        35       80.46     2816.10
1  MC-100001        1  FN-11        30       35.14     1054.20
2  MC-100001        2  PL-31        34       48.81     1659.54
3  MC-100002        1  ST-01        19      768.04    14592.76
4  MC-100002        2  PL-31         6      454.52     2727.12
5  MC-100002        3  WR-14        34      873.47    29697.98
6013 2400


In [31]:
merged = invoices.merge(line_items, on="invoice_id", how="left")
print(len(merged))
print(f"{merged['amount'].sum():,.2f}")

6013
216,593,613.75


In [32]:
print(f"{line_items['line_total'].sum():,.2f}")

72,448,297.35


## Align: Standardizing Values to a Canonical Form

In [33]:
print(invoices["vendor_name"].nunique())

32


In [34]:
def normalize(name):
    name = re.sub(r"\s+", " ", name.strip())
    return name.casefold()

print(invoices["vendor_name"].map(normalize).nunique())

25


In [35]:
def normalize(name):
    name = re.sub(r"[.,]", "", name)
    name = re.sub(r"\s+", " ", name.strip())
    return name.casefold()

print(invoices["vendor_name"].map(normalize).nunique())

17


In [36]:
canonical = {
    "nordic steel": "Nordic Steel Co.",
    "nordic steel co": "Nordic Steel Co.",
    "kwan precision": "Kwan Precision Ltd.",
    "kwan precision ltd": "Kwan Precision Ltd.",
    "alpha fasteners": "Alpha Fasteners Inc.",
    "alpha fasteners inc": "Alpha Fasteners Inc.",
    "rhein polymer": "Rhein Polymer GmbH",
    "rhein polymer gmbh": "Rhein Polymer GmbH",
    "meridian tooling": "Meridian Tooling (internal)",
    "meridian tooling internal": "Meridian Tooling (internal)",
    "meridian tooling (internal)": "Meridian Tooling (internal)",
    "sato industrial": "Sato Industrial K.K.",
    "sato industrial kk": "Sato Industrial K.K.",
    "perez castings": "Perez Castings S.A.",
    "perez castings sa": "Perez Castings S.A.",
    "britewire": "Britewire Ltd.",
    "britewire ltd": "Britewire Ltd.",
}

normalized = invoices["vendor_name"].map(normalize)
invoices["vendor"] = normalized.map(canonical)

print(invoices["vendor"].isnull().sum())
print(invoices["vendor"].nunique())

0
8


In [37]:
print(sorted(invoices["currency"].unique()))

currency_map = {"usd": "USD", "us dollar": "USD",
                "eur": "EUR", "euro": "EUR"}
invoices["currency"] = (
    invoices["currency"].str.strip().str.casefold().map(currency_map)
)

print(sorted(invoices["currency"].unique()))

['EUR', 'Euro', 'US Dollar', 'USD', 'eur', 'usd']
['EUR', 'USD']


### Currency conversion (live FX)

The next cell is the one **live** cell in the notebook: it retrieves historical daily EUR→USD rates from the European Central Bank via the Frankfurter API, exactly as in Chapter 6. Because these are *historical* rates for a closed quarter, the values are stable across runs.

If the network is unavailable, the cell falls back to documented illustrative rates so the rest of the notebook still executes; the printed total then matches the chapter's illustrative figure. When run with network access, it uses the real ECB rates.

In [38]:
import requests

# Documented illustrative monthly rates (used as fallback only).
ILLUSTRATIVE_MONTHLY = pd.Series({7: 1.0850, 8: 1.0920, 9: 1.0975},
                                 name="USD")

try:
    # Historical daily EUR->USD from the ECB via Frankfurter, Q3 2026.
    url = "https://api.frankfurter.dev/v1/2026-07-01..2026-09-30"
    resp = requests.get(url, params={"base": "EUR", "symbols": "USD"})
    resp.raise_for_status()
    rates = resp.json()["rates"]
    daily = pd.Series(
        {pd.Timestamp(d): r["USD"] for d, r in rates.items()}
    ).sort_index()
    monthly_rate = daily.groupby(daily.index.month).mean()
    monthly_rate.name = "USD"
    print("Live ECB rates retrieved.")

except Exception as e:
    print(f"Live call unavailable ({type(e).__name__}); using illustrative rates.")
    monthly_rate = ILLUSTRATIVE_MONTHLY.copy()

print(monthly_rate.round(4))

Live ECB rates retrieved.
7    1.1417
8    1.1593
9    1.1585
Name: USD, dtype: float64


In [39]:
invoices["month"] = invoices["invoice_date"].dt.month

def to_usd(row):
    if row["currency"] == "EUR":
        return round(row["amount"] * monthly_rate[row["month"]], 2)
    return row["amount"]

invoices["amount_usd"] = invoices.apply(to_usd, axis=1)

print(f"{invoices['amount_usd'].sum():,.2f}")

77,262,370.37


In [40]:
eur = invoices[invoices["currency"] == "EUR"]
weights = eur.groupby("month")["amount"].sum()

weighted = np.average(monthly_rate, weights=weights)
simple = monthly_rate.mean()

print(f"volume-weighted rate: {weighted:.4f}")
print(f"simple mean rate:     {simple:.4f}")
print(f"difference:           {abs(weighted - simple) / simple:.4%}")

volume-weighted rate: 1.1531
simple mean rate:     1.1532
difference:           0.0114%


## Prove: Validation and Reconciliation

In [41]:
assert invoices["invoice_id"].is_unique, "duplicate invoice IDs remain"
assert invoices["amount"].notnull().all(), "null amounts remain"
assert (invoices["amount"] > 0).all(), "non-positive amounts present"

invoice_ids = set(invoices["invoice_id"])
line_ids = set(line_items["invoice_id"])
assert invoice_ids.issubset(line_ids), "invoices without line items"

print("All checks passed.")

All checks passed.


In [42]:
control = pd.read_csv("meridian_control_totals.csv")
print(control)

       channel   true_spend
0  email_sheet  13400706.52
1    erp_north  29704765.67
2    erp_south  29342825.16


In [43]:
cleaned = invoices.groupby("channel")["amount"].sum()

reconciliation = control.set_index("channel").join(
    cleaned.rename("cleaned")
)
reconciliation["difference"] = (
    reconciliation["cleaned"] - reconciliation["true_spend"]
).round(2)
print(reconciliation)

              true_spend      cleaned  difference
channel                                          
email_sheet  13400706.52  13400706.52         0.0
erp_north    29704765.67  29704765.67         0.0
erp_south    29342825.16  29342825.16         0.0


In [44]:
spend_by_vendor = (
    invoices.groupby("vendor")["amount_usd"].sum()
    .sort_values(ascending=False)
)
print(spend_by_vendor.round(2))

vendor
Nordic Steel Co.               23746909.45
Perez Castings S.A.            18034981.42
Sato Industrial K.K.           13440385.42
Rhein Polymer GmbH             11547902.10
Kwan Precision Ltd.             4773154.38
Meridian Tooling (internal)     3537678.65
Alpha Fasteners Inc.            1210904.19
Britewire Ltd.                   970454.76
Name: amount_usd, dtype: float64


---

**Reconciliation note.** The external reconciliation (Code Example 7.27) compares cleaned spend against control totals in *native currency*, before the FX conversion — because the control totals are themselves in native currency. The currency-converted grand total (7.23) is a separate analytical layer on top of the proven cleaning.